# ✅ Write-Audit-Publish (WAP) — Safe Data Ingestion with Iceberg Branches

The WAP pattern lets you **stage data on a branch**, run quality checks, and only **publish to production/main branch** if everything passes — all without copying data or locking the table.

```
                        Write-Audit-Publish Flow
───────────────────────────────────────────────────────────────────

  main branch (production)         staging branch (isolated)
  ┌──────────────────────┐         ┌──────────────────────────┐
  │ Snapshot A           │         │ Snapshot A (forked)      │
  │  (users read this)   │         │  + new staged data       │
  └──────────┬───────────┘         └──────────┬───────────────┘
             │                                │
             │          ┌──────────┐          │
             │          │  AUDIT   │◄─────────┘
             │          │ (quality │
             │          │  checks) │
             │          └────┬─────┘
             │               │
             │          Pass? │ Fail?
             │          ┌────┴─────┐
             │     ┌────▼───┐  ┌───▼────┐
             │     │PUBLISH │  │DISCARD │
             │     │fast-fwd│  │drop    │
             │     └────┬───┘  │branch  │
             │          │      └────────┘
  ┌──────────▼──────────▼┐
  │ Snapshot B            │
  │  (new production)     │
  └───────────────────────┘
```

| Step | What Happens | Pyspark Mechanism |
|------|-------------|-------------------|
| **Write** | Ingest data to isolated branch | `spark.conf.set('spark.wap.branch', 'branch')` + standard `INSERT` |
| **Audit** | Run quality checks on branch | `SELECT ... VERSION AS OF 'branch'` |
| **Publish** | Fast-forward main → branch | `CALL catalog.system.fast_forward('table', 'main', 'branch')` |

**Prerequisites:** Run `setup.ipynb` first.

---
## ⚙️ Connect to Apache Spark + Polaris

In [ ]:
import os
import pyspark
from pyspark.sql import SparkSession

# Spark requires you to add Iceberg runtime libraries and AWS libraries to talk to S3.
# `hadoop-aws` allows Spark to communicate with S3 (SeaweedFS in our case).
builder = SparkSession.builder \
    .appName("WAP-Iceberg") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,org.apache.iceberg:iceberg-aws-bundle:1.5.0") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.lakehouse", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.lakehouse.type", "rest") \
    .config("spark.sql.catalog.lakehouse.uri", "http://polaris:8181/api/catalog") \
    .config("spark.sql.catalog.lakehouse.credential", "root:polaris-secret") \
    .config("spark.sql.catalog.lakehouse.scope", "PRINCIPAL_ROLE:ALL") \
    .config("spark.sql.catalog.lakehouse.warehouse", "lakehouse") \
    .config("spark.sql.catalog.lakehouse.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.lakehouse.s3.endpoint", "http://seaweedfs:8333") \
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true") \
    .config("spark.sql.catalog.lakehouse.s3.access-key-id", "lakehouse-admin") \
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", "lakehouse-secret-key")

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

def run_query(sql, display=True):
    """Execute a Spark SQL query and display results."""
    try:
        df = spark.sql(sql)
        if display and not df.isEmpty():
            df.show(truncate=False)
        return df.collect()
    except Exception as e:
        print(f"❌ Error: {e}")
        return []


print("✅ Connected to PySpark + Polaris")

---
# Part 1 — Setting Up the Production Table

We'll create a `transactions` table in the silver layer representing a production financial dataset. This is the table that downstream dashboards and reports read from — we can't afford to publish bad data here.

In [ ]:
run_query("DROP TABLE IF EXISTS lakehouse.silver.transactions")

run_query("""
CREATE TABLE lakehouse.silver.transactions (
    txn_id      BIGINT,
    account_id  STRING,
    amount      DOUBLE,
    currency    STRING,
    txn_date    DATE,
    status      STRING
) USING iceberg
TBLPROPERTIES ('write.wap.enabled'='true')
""")
print("✅ Table 'transactions' created in silver layer")

### 📥 Seed Production Data

Load a baseline of known-good transactions. This represents the current production state.

In [ ]:
run_query("""
INSERT INTO lakehouse.silver.transactions VALUES
    (1, 'ACC-001', 1500.00, 'USD', DATE '2026-03-01', 'completed'),
    (2, 'ACC-002',  250.75, 'USD', DATE '2026-03-01', 'completed'),
    (3, 'ACC-001',  890.00, 'EUR', DATE '2026-03-02', 'completed'),
    (4, 'ACC-003', 3200.00, 'USD', DATE '2026-03-02', 'completed'),
    (5, 'ACC-002',  175.50, 'GBP', DATE '2026-03-03', 'completed')
""")
print("✅ 5 baseline transactions loaded")

In [ ]:
print("📋 Production data (main branch):")
print()
run_query("SELECT * FROM lakehouse.silver.transactions ORDER BY txn_id");

---
# Part 2 — The WAP Pattern: Happy Path 🟢

A new batch of transactions arrives from an upstream system. Before publishing to production, we need to ensure data quality. Here's the full Write → Audit → Publish cycle.

## 2.1 — Write: Create a Staging Branch

Create an isolated branch forked from `main`. This is a **metadata-only** operation — no data is copied. The branch starts at the same snapshot as `main`.

In [ ]:
run_query("ALTER TABLE lakehouse.silver.transactions CREATE BRANCH audit_batch_01")
print("✅ Branch 'audit_batch_01' created — forked from main")

In [ ]:
print("📋 Table refs — main + new branch share the same snapshot:")
print()
run_query("""
SELECT name, type, snapshot_id
FROM lakehouse.silver.transactions.refs
""");

### 📥 Write New Data to the Staging Branch

Iceberg WAP allows you to direct inserts to a branch simply by setting the `spark.wap.branch` property. Any standard `INSERT` will then route only to that branch!

In [ ]:
spark.conf.set('spark.wap.branch', 'audit_batch_01')

run_query("""
INSERT INTO lakehouse.silver.transactions
VALUES
    (6, 'ACC-001',  420.00, 'USD', DATE '2026-03-04', 'completed'),
    (7, 'ACC-004', 1100.00, 'EUR', DATE '2026-03-04', 'completed'),
    (8, 'ACC-002',  330.25, 'USD', DATE '2026-03-05', 'completed'),
    (9, 'ACC-003',  750.00, 'GBP', DATE '2026-03-05', 'completed')
""")

# Important: When we are done writing to the branch, unset WAP so future queries don't default to the branch
spark.conf.unset('spark.wap.branch')

print("✅ 4 new transactions written to staging branch")

### 🔍 Verify Branch Isolation

The key feature: production (`main`) and staging (`audit_batch_01`) show **different data**. Downstream consumers are completely unaffected.

In [ ]:
print("📋 Production (main) — still 5 rows, unchanged:")
prod_rows = run_query("SELECT count(*) AS row_count FROM lakehouse.silver.transactions", display=False)
print(f"Main branch rows: {prod_rows[0]['row_count']}")

print()
print("📋 Staging branch — 9 rows (5 original + 4 new):")
stage_rows = run_query("""
SELECT count(*) AS row_count 
FROM lakehouse.silver.transactions VERSION AS OF 'audit_batch_01'
""", display=False)
print(f"Audit branch rows: {stage_rows[0]['row_count']}")

print()
print(f"🔒 Branch isolation confirmed!")

In [ ]:
print("📋 Refs after write — staging branch has advanced to a new snapshot:")
print()
run_query("""
SELECT name, type, snapshot_id
FROM lakehouse.silver.transactions.refs
""");

## 2.2 — Audit: Data Quality Checks on the Branch

Now we run validation queries **against the staging branch only**. These checks ensure the new batch meets our data quality requirements before it reaches production.

| Check | Rule | Threshold |
|-------|------|-----------|
| No NULL amounts | `amount IS NOT NULL` | 0 violations |
| No negative amounts | `amount > 0` | 0 violations |
| Valid currency codes | `currency IN ('USD','EUR','GBP')` | 0 violations |
| No duplicate txn_ids | `COUNT(DISTINCT txn_id) = COUNT(*)` | exact match |

In [ ]:
def audit_branch(branch_name):
    """Run data quality checks against a staging branch. Returns True if all pass."""
    checks = [
        (
            "No NULL amounts",
            f"""
            SELECT count(*) AS violations
            FROM lakehouse.silver.transactions VERSION AS OF '{branch_name}'
            WHERE amount IS NULL
            """,
            lambda rows: rows[0]['violations'] == 0,
        ),
        (
            "No negative amounts",
            f"""
            SELECT count(*) AS violations
            FROM lakehouse.silver.transactions VERSION AS OF '{branch_name}'
            WHERE amount <= 0
            """,
            lambda rows: rows[0]['violations'] == 0,
        ),
        (
            "Valid currency codes",
            f"""
            SELECT count(*) AS violations
            FROM lakehouse.silver.transactions VERSION AS OF '{branch_name}'
            WHERE currency NOT IN ('USD', 'EUR', 'GBP')
            """,
            lambda rows: rows[0]['violations'] == 0,
        ),
        (
            "No duplicate txn_ids",
            f"""
            SELECT
                count(*) AS total,
                count(DISTINCT txn_id) AS distinct_ids
            FROM lakehouse.silver.transactions VERSION AS OF '{branch_name}'
            """,
            lambda rows: rows[0]['total'] == rows[0]['distinct_ids'],
        ),
    ]

    all_passed = True
    for name, sql, check_fn in checks:
        result = run_query(sql, display=False)
        passed = check_fn(result)
        status = "✅ PASS" if passed else "❌ FAIL"
        print(f"  {status} — {name}")
        if not passed:
            all_passed = False

    return all_passed


print(f"🔍 Auditing branch 'audit_batch_01'...")
print()
passed = audit_branch("audit_batch_01")
print()
print(f"{'🟢 All checks passed — safe to publish!' if passed else '🔴 Audit failed — do NOT publish!'}")

## 2.3 — Publish: Fast-Forward Main to the Branch

All checks passed! Now we **fast-forward** `main` to point to the staging branch's latest snapshot via native Spark SQL operations. This is:
- **Atomic** — readers instantly see all new data or none of it
- **Metadata-only** — no data files are moved or copied
- **Instantaneous** — regardless of data size

In [ ]:
run_query("CALL lakehouse.system.fast_forward('lakehouse.silver.transactions', 'main', 'audit_batch_01')")
print("✅ Published! main branch fast-forwarded to audit_batch_01")

In [ ]:
print("📋 Production data after publish — all 9 transactions now visible:")
print()
run_query("SELECT * FROM lakehouse.silver.transactions ORDER BY txn_id");

In [ ]:
print("📋 Refs after publish — main and branch now point to the same snapshot:")
print()
run_query("""
SELECT name, type, snapshot_id
FROM lakehouse.silver.transactions.refs
""");

### 🧹 Clean Up the Staging Branch

After a successful publish, the staging branch is no longer needed. Drop it to keep metadata clean.

In [ ]:
run_query("ALTER TABLE lakehouse.silver.transactions DROP BRANCH audit_batch_01")
print("✅ Branch 'audit_batch_01' dropped — publish complete")

---
# Part 3 — The WAP Pattern: Rejected Batch 🔴

Now let's see what happens when bad data arrives. The audit step catches the issues, and we **discard** the branch — production is never affected.

## 3.1 — Write: Stage a Bad Batch

In [ ]:
run_query("ALTER TABLE lakehouse.silver.transactions CREATE BRANCH audit_batch_02")
print("✅ Branch 'audit_batch_02' created")

In [ ]:
spark.conf.set('spark.wap.branch', 'audit_batch_02')

run_query("""
INSERT INTO lakehouse.silver.transactions
VALUES
    (10, 'ACC-001',  500.00, 'USD', DATE '2026-03-06', 'completed'),
    (11, 'ACC-005', -200.00, 'USD', DATE '2026-03-06', 'completed'),
    (12, 'ACC-002',  680.00, 'XYZ', DATE '2026-03-06', 'completed'),
    (10, 'ACC-001',  500.00, 'USD', DATE '2026-03-06', 'completed')
""")

spark.conf.unset('spark.wap.branch')

print("✅ 4 transactions written to staging (includes bad data!)")
print("   ⚠️  txn 11 has negative amount")
print("   ⚠️  txn 12 has invalid currency 'XYZ'")
print("   ⚠️  txn 10 is duplicated")

## 3.2 — Audit: Checks Fail

In [ ]:
print(f"🔍 Auditing branch 'audit_batch_02'...")
print()
passed = audit_branch("audit_batch_02")
print()
print(f"{'🟢 All checks passed — safe to publish!' if passed else '🔴 Audit FAILED — batch will be discarded!'}")

### 🔍 Inspect the Violations

Let's see exactly what failed — querying the branch to understand the issues before discarding it.

In [ ]:
print("❌ Negative amounts:")
run_query("""
SELECT txn_id, account_id, amount
FROM lakehouse.silver.transactions VERSION AS OF 'audit_batch_02'
WHERE amount <= 0
""")

print()
print("❌ Invalid currencies:")
run_query("""
SELECT txn_id, account_id, currency
FROM lakehouse.silver.transactions VERSION AS OF 'audit_batch_02'
WHERE currency NOT IN ('USD', 'EUR', 'GBP')
""")

print()
print("❌ Duplicate txn_ids:")
run_query("""
SELECT txn_id, count(*) AS occurrences
FROM lakehouse.silver.transactions VERSION AS OF 'audit_batch_02'
GROUP BY txn_id
HAVING count(*) > 1
""");

## 3.3 — Discard: Drop the Failed Branch

Since the audit failed, we simply **drop the branch**. Production data on `main` is completely unaffected — as if the bad batch never existed.

In [ ]:
run_query("ALTER TABLE lakehouse.silver.transactions DROP BRANCH audit_batch_02")
print("🗑️  Branch 'audit_batch_02' dropped — bad batch discarded")

In [ ]:
print("📋 Production data — still exactly 9 rows, untouched:")
print()
run_query("SELECT * FROM lakehouse.silver.transactions ORDER BY txn_id");

In [ ]:
print("📋 Only 'main' ref remains — staging branch is gone:")
print()
run_query("""
SELECT name, type, snapshot_id
FROM lakehouse.silver.transactions.refs
""");

---
# Part 4 — Under the Hood: Snapshots & Branches

Let's inspect the Iceberg metadata to understand what happened at the snapshot level throughout the entire WAP lifecycle.

In [ ]:
print("📸 Full snapshot history — shows the complete WAP lifecycle:")
print()
run_query("""
SELECT committed_at, snapshot_id, parent_id, operation
FROM lakehouse.silver.transactions.snapshots
ORDER BY committed_at
""");

In [ ]:
print("📁 Data files — branches share files, no duplication:")
print()
run_query("""
SELECT
    record_count,
    file_size_in_bytes,
    file_format
FROM lakehouse.silver.transactions.files
""");

---
## 📊 Summary

| Concept | Implementation | Key Benefit |
|---------|---------------|-------------|
| **Write** | `CREATE BRANCH` + `spark.wap.branch` | Isolated staging, zero duplication |
| **Audit** | `SELECT ... VERSION AS OF` with validation logic | Query staged data without affecting production |
| **Publish** | `CALL system.fast_forward(main, branch)` | Atomic, metadata-only promotion |
| **Reject** | `DROP BRANCH` | Production untouched, instant cleanup |

```
WAP vs Traditional Staging
──────────────────────────────────────────────────────────────────
                        Traditional          WAP + Branching
  Data duplication      Full copy to         Zero — shared files
                        staging table
  Publish               INSERT + DELETE      Metadata pointer swap
  Rollback              Manual cleanup       DROP BRANCH
  Isolation             Separate table       Same table, diff branch
  Atomicity             Multi-step           Single fast-forward
```


---
## 🧹 Cleanup (Optional)

In [ ]:
# Uncomment to drop the table:
# run_query("DROP TABLE IF EXISTS lakehouse.silver.transactions")
# print("🗑️ Table 'transactions' dropped")